In [3]:
import numpy
import random
import pandas
import time
from deap import creator
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import matthews_corrcoef
from scipy.stats import pointbiserialr

from training_config import TrainingConfig
from multi_objective_training import MultiObjectiveTraining
from single_objective_training import SingleObjectiveTraining

AttributeError: module 'deap.creator' has no attribute 'Individual'

In [ ]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    numpy.random.seed(seed)

In [ ]:
SEED: int = 42

CSV_TRAIN_PATH: str = "hmeq/hmeq_preprocessed_train_data.csv"
CSV_TEST_PATH: str = "hmeq/hmeq_preprocessed_test_data.csv"
TARGET_COLUMN: str = "bad"

RESULT_PATH: str = time.strftime("%Y-%m-%d_%H-%M-%S")
RESULT_PATH_MULTI: str = f"{RESULT_PATH}_multi"
RESULT_PATH_SINGLE: str = f"{RESULT_PATH}_single"

In [ ]:
# Load train set
df_train: pandas.DataFrame = pandas.read_csv(CSV_TRAIN_PATH)

# Split data into training and validation sets
X_search_pandas: pandas.DataFrame = df_train.drop(columns=[TARGET_COLUMN])
y_search_pandas: pandas.Series = df_train[TARGET_COLUMN]

X_search: numpy.ndarray = numpy.ascontiguousarray(X_search_pandas.to_numpy(), dtype=numpy.float32)
y_search: numpy.ndarray = numpy.ascontiguousarray(y_search_pandas.to_numpy(), dtype=numpy.float32)

feature_names: list[str] = list(X_search_pandas.columns)

cv: StratifiedKFold = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

In [ ]:
# Load test set
df_test: pandas.DataFrame = pandas.read_csv(CSV_TEST_PATH)

y_test: pandas.Series = df_test[TARGET_COLUMN]
X_test: pandas.DataFrame = df_test.drop(columns=[TARGET_COLUMN])

In [ ]:
# Pre-compute folds and scaling
fold_indices: list[tuple[numpy.ndarray, numpy.ndarray]] = list(cv.split(X_search, y_search))

X_train_scaled_folds: list[numpy.ndarray] = []
X_val_scaled_folds: list[numpy.ndarray] = []
y_train_folds: list[numpy.ndarray] = []
y_val_folds: list[numpy.ndarray] = []

n_splits: int = len(fold_indices)
n_features: int = len(feature_names)
corr_matrix: numpy.ndarray = numpy.zeros((n_splits, n_features), dtype=float)
for fold_idx, (train_idx, val_idx) in enumerate(fold_indices):
    X_fold_train: numpy.ndarray = X_search[train_idx]
    X_fold_val: numpy.ndarray = X_search[val_idx]
    y_fold_train: numpy.ndarray = y_search[train_idx]
    y_fold_val: numpy.ndarray = y_search[val_idx]

    # Pre-scale (all features at once)
    scaler: StandardScaler = StandardScaler()
    X_train_scaled_folds.append(scaler.fit_transform(X_fold_train))
    X_val_scaled_folds.append(scaler.transform(X_fold_val))
    y_train_folds.append(y_fold_train)
    y_val_folds.append(y_fold_val)

    # Calculate Point-biserial correlation coefficients (target value is binary and the input variables are continuous)
    # Calculate the average correlation for each feature across all fold
    for col_idx in range(X_fold_train.shape[1]):
        feature_col: numpy.ndarray = X_fold_train[:, col_idx]
        unique_vals: int = len(numpy.unique(feature_col))

        if unique_vals <= 1:
            corr: float = 0.0
        elif unique_vals == 2:
            corr: float = matthews_corrcoef(y_fold_train, feature_col)
        else:
            corr, _ = pointbiserialr(y_fold_train, feature_col)

        corr_matrix[fold_idx, col_idx] = corr

In [ ]:
training_results_multi: dict[int, list[list[creator.Individual]]] = {}
training_results_single: dict[int, list[creator.IndividualSingle]] = {}

seeds: list[int] = [42, 123]
for s in seeds:
    set_seed(s)
    multi_objective_training: MultiObjectiveTraining = MultiObjectiveTraining(
        config=TrainingConfig(seed=s, result_directory=RESULT_PATH_MULTI),
        feature_names=feature_names,
        fold_indices=fold_indices,
        X_train_scaled_folds=X_train_scaled_folds,
        X_val_scaled_folds=X_val_scaled_folds,
        y_train_folds=y_train_folds,
        y_val_folds=y_val_folds,
        corr_matrix=corr_matrix,
    )
    pareto_front: list[creator.Individual] = multi_objective_training.run()
    training_results_multi[s] = pareto_front
    multi_objective_training.clear_cache()

    set_seed(s)
    single_objective_training: SingleObjectiveTraining = SingleObjectiveTraining(
        config=TrainingConfig(seed=s, result_directory=RESULT_PATH_SINGLE),
        feature_names=feature_names,
        fold_indices=fold_indices,
        X_train_scaled_folds=X_train_scaled_folds,
        X_val_scaled_folds=X_val_scaled_folds,
        y_train_folds=y_train_folds,
        y_val_folds=y_val_folds)
    best_indi: creator.IndividualSingle = single_objective_training.run()
    training_results_single[s] = best_indi
    single_objective_training.clear_cache()